In [ ]:
import torch
from torchvision import models, transforms
import requests
from PIL import Image
import torch.nn.functional as F
import matplotlib.pyplot as plt

# define alexnet model
alexnet = models.alexnet(pretrained=True).cuda(0)

# get labels with error handling
try:
    labels = {int(key):value for (key, value) in requests.get('https://s3.amazonaws.com/mlpipes/pytorch-quick-start/labels.json').json().items()}
except:
    # use alternative source
    response = requests.get('https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json')
    simple_labels = response.json()
    labels = {i: label for i, label in enumerate(simple_labels)}

# transform image for use in model
preprocess = transforms.Compose([
   transforms.Resize(256),
   transforms.CenterCrop(224),
   transforms.ToTensor(),
   transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
])

# load the image from its url
url = 'https://post.medicalnewstoday.com/wp-content/uploads/sites/3/2020/02/322868_1100-732x549.jpg'
img = Image.open(requests.get(url, stream=True).raw)
img_t = preprocess(img).unsqueeze_(0).cuda(0)
print(img_t.shape)

# classify the image with alexnet
scores, class_idx = alexnet(img_t).max(1)
print('Predicted class:', labels[class_idx.item()])

# extract weights
w0 = alexnet.features[0].weight.data
w1 = alexnet.features[3].weight.data
w2 = alexnet.features[6].weight.data
w3 = alexnet.features[8].weight.data
w4 = alexnet.features[10].weight.data
w5 = alexnet.classifier[1].weight.data
w6 = alexnet.classifier[4].weight.data
w7 = alexnet.classifier[6].weight.data
w = [w0,w1,w2,w3,w4,w5,w6,w7]

# save and load weights
torch.save(w, 'Hahn_Alex.pt')
w = torch.load('Hahn_Alex.pt')
[w0,w1,w2,w3,w4,w5,w6,w7] = w

print(f"img_t shape: {img_t.shape}, w0 shape: {w0.shape}")

# visualization functions
def scale(img):
    # Normalize the NumPy array to the range [0, 1]
    max_value = img.max()
    min_value = img.min()
    normalized_array = (img - min_value) / (max_value - min_value)
    return normalized_array

def tensor_plot(img_t, index=0):
    numpy_array = img_t[index,:,:,:].cpu().numpy()
    numpy_array_transposed = numpy_array.transpose(1, 2, 0)
    numpy_array_transposed = scale(numpy_array_transposed)
    plt.imshow(numpy_array_transposed)
    plt.show()

# visualize input image
tensor_plot(img_t)

# visualize some filters
print("Visualizing first few conv1 filters...")
for i in range(4):  # show first 4 filters instead of all 64
    tensor_plot(w0,i)

# manual forward pass
f0 = F.conv2d(img_t, w0, stride=4, padding=2)
print(f"f0 shape: {f0.shape}")

# visualize conv1 output
plt.figure(figsize=(10, 5))
for i in range(8):  # show first 8 channels
    plt.subplot(2, 4, i+1)
    plt.imshow(f0[0,i,:,:].cpu().numpy(), cmap='viridis')
    plt.title(f'Ch {i}')
    plt.axis('off')
plt.suptitle('Conv1 Output (first 8 channels)')
plt.show()

# continue forward pass
f1 = F.relu(f0)
f2 = F.max_pool2d(f1, kernel_size=3, stride=2, padding=0, dilation=1)
print(f"After MaxPool1: {f2.shape}")

f3 = F.conv2d(f2, w1, stride=1, padding=2)
f4 = F.relu(f3)
f5 = F.max_pool2d(f4, kernel_size=3, stride=2, padding=0, dilation=1)
print(f"After MaxPool2: {f5.shape}")

f6 = F.conv2d(f5, w2, stride=1, padding=1)
f7 = F.relu(f6)
f8 = F.conv2d(f7, w3, stride=1, padding=1)
f9 = F.relu(f8)
f10 = F.conv2d(f9, w4, stride=1, padding=1)
f11 = F.relu(f10)
f12 = F.max_pool2d(f11, kernel_size=3, stride=2, padding=0, dilation=1)
print(f"After MaxPool3: {f12.shape}")

# classifier layers
f13 = F.adaptive_avg_pool2d(f12, output_size=6).flatten(1)  # fixed: flatten(1) to keep batch dim
print(f"After adaptive pool + flatten: {f13.shape}")

f14 = F.linear(f13, w5)
f15 = F.relu(f14)
f16 = F.linear(f15, w6)
f17 = F.relu(f16)
f18 = F.linear(f17, w7)
print(f"Final output shape: {f18.shape}")

# get prediction
out = f18.argmax().item()
print(f"Manual forward pass prediction: {labels[out]}")

# visualize output distribution
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(f18[0].cpu().detach().numpy())
plt.title('Raw Logits')
plt.xlabel('Class Index')
plt.ylabel('Logit Value')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
probs = F.softmax(f18, dim=1)[0].cpu().detach().numpy()
top10_idx = probs.argsort()[-10:][::-1]
top10_labels = [labels[idx].split(',')[0][:15] for idx in top10_idx]
top10_probs = probs[top10_idx]

plt.barh(range(10), top10_probs)
plt.yticks(range(10), top10_labels)
plt.xlabel('Probability')
plt.title('Top 10 Predictions')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# compare with alexnet prediction
print(f"\nAlexNet direct prediction: {labels[class_idx.item()]}")
print(f"Manual forward prediction: {labels[out]}")

In [ ]:
import torch
from torchvision import models, transforms
import requests
from PIL import Image
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# setup
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
alexnet = models.alexnet(pretrained=True).to(device)
alexnet.eval()

# ImageNet class labels - using working URL
def get_imagenet_labels():
    """Get ImageNet labels from various sources"""
    urls = [
        'https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json',
        'https://gist.githubusercontent.com/yrevar/942d3a0ac09ec9e5eb3a/raw/238f720ff059c1f82f368259d1ca4ffa5dd8f9f5/imagenet1000_clsidx_to_labels.txt'
    ]

    # Try first URL (simple labels)
    try:
        response = requests.get(urls[0])
        simple_labels = response.json()
        return {i: label for i, label in enumerate(simple_labels)}
    except:
        pass

    # Try second URL (detailed labels)
    try:
        response = requests.get(urls[1])
        labels = {}
        for line in response.text.strip().split('\n'):
            parts = line.split(': ')
            if len(parts) == 2:
                idx = int(parts[0])
                label = parts[1].strip("',")
                labels[idx] = label
        return labels
    except:
        pass

    # Final fallback
    print("Warning: Could not load ImageNet labels, using generic labels")
    return {i: f'class_{i}' for i in range(1000)}

# get labels
labels = get_imagenet_labels()
print(f"Loaded {len(labels)} labels")

# transform
preprocess = transforms.Compose([
   transforms.Resize(256),
   transforms.CenterCrop(224),
   transforms.ToTensor(),
   transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# extract weights
w0 = alexnet.features[0].weight.data
w1 = alexnet.features[3].weight.data
w2 = alexnet.features[6].weight.data
w3 = alexnet.features[8].weight.data
w4 = alexnet.features[10].weight.data
w5 = alexnet.classifier[1].weight.data
w6 = alexnet.classifier[4].weight.data
w7 = alexnet.classifier[6].weight.data
w = [w0,w1,w2,w3,w4,w5,w6,w7]

# layer functions
def C1(x,w):
    return F.conv2d(x, w, stride=4, padding=2)
def R(x):
    return F.relu(x)
def M(x):
    return F.max_pool2d(x, kernel_size=3, stride=2, padding=0, dilation=1)
def C2(x,w):
    return F.conv2d(x, w, stride=1, padding=2)
def C3(x,w):
    return F.conv2d(x, w, stride=1, padding=1)
def A(x):
    return F.adaptive_avg_pool2d(x, output_size=6).flatten(1)
def L(x,w):
    return F.linear(x,w)

# manual forward
def forward(img_t, weights):
    w0, w1, w2, w3, w4, w5, w6, w7 = weights

    f0 = C1(img_t, w0)
    f1 = R(f0)
    f2 = M(f1)
    f3 = C2(f2, w1)
    f4 = R(f3)
    f5 = M(f4)
    f6 = C3(f5, w2)
    f7 = R(f6)
    f8 = C3(f7, w3)
    f9 = R(f8)
    f10 = C3(f9, w4)
    f11 = R(f10)
    f12 = M(f11)
    f13 = A(f12)
    f14 = L(f13, w5)
    f15 = R(f14)
    f16 = L(f15, w6)
    f17 = R(f16)
    f18 = L(f17, w7)

    out = f18.argmax().item()
    print(f"Manual forward prediction: {labels[out]}")

    return f18

# plot channels
def plot_channels(tensor, n=64, title=""):
    if tensor.is_cuda:
        tensor = tensor.cpu()
    if len(tensor.shape) == 4:
        tensor = tensor[0]

    data = tensor.detach().numpy()
    n_show = min(n, data.shape[0])

    cols = 8
    rows = (n_show + cols - 1) // cols

    plt.figure(figsize=(16, 2*rows))
    for i in range(n_show):
        plt.subplot(rows, cols, i+1)
        plt.imshow(data[i], cmap='viridis')
        plt.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# viz all layers
def viz_all_layers(img_t):
    names = ['Conv1', 'ReLU1', 'MaxPool1', 'Conv2', 'ReLU2', 'MaxPool2',
             'Conv3', 'ReLU3', 'Conv4', 'ReLU4', 'Conv5', 'ReLU5', 'MaxPool3']

    for i in range(13):
        with torch.no_grad():
            out = alexnet.features[:i+1](img_t)
        plot_channels(out, 32 if i < 6 else 64, f"Layer {i}: {names[i]}")

# main test
if __name__ == "__main__":
    # load image
    url = 'https://post.medicalnewstoday.com/wp-content/uploads/sites/3/2020/02/322868_1100-732x549.jpg'
    img = Image.open(requests.get(url, stream=True).raw)
    img_t = preprocess(img).unsqueeze(0).to(device)

    # test alexnet
    out = alexnet(img_t)
    pred = out.argmax(1).item()
    print(f"AlexNet prediction: {labels[pred]}")

    # test manual forward
    manual_out = forward(img_t, w)

    # viz layers
    print("\nVisualizing all layers...")
    viz_all_layers(img_t)

# Extras

In [ ]:
import torch
from torchvision import models, transforms
import requests
from PIL import Image
import torch.nn.functional as F

#define alexnet model
alexnet = models.alexnet(pretrained=True).cuda(0)
labels = {int(key):value for (key, value) in requests.get('https://s3.amazonaws.com/mlpipes/pytorch-quick-start/labels.json').json().items()}

#transform image for use in model
preprocess = transforms.Compose([
   transforms.Resize(256),
   transforms.CenterCrop(224),
   transforms.ToTensor(),
   transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
])

#load the image from its url
url = 'https://post.medicalnewstoday.com/wp-content/uploads/sites/3/2020/02/322868_1100-732x549.jpg'
img = Image.open(requests.get(url, stream=True).raw)
img_t = preprocess(img).unsqueeze_(0).cuda(0)

In [ ]:
img_t.shape

In [ ]:
labels

In [ ]:
#classify the image with alexnet
scores, class_idx = alexnet(img_t).max(1)
print('Predicted class:', labels[class_idx.item()])

In [ ]:
# labels

In [ ]:
w0 = alexnet.features[0].weight.data
w1 = alexnet.features[3].weight.data
w2 = alexnet.features[6].weight.data
w3 = alexnet.features[8].weight.data
w4 = alexnet.features[10].weight.data
w5 = alexnet.classifier[1].weight.data
w6 = alexnet.classifier[4].weight.data
w7 = alexnet.classifier[6].weight.data

In [ ]:
w = [w0,w1,w2,w3,w4,w5,w6,w7]

In [ ]:
torch.save(w, 'Hahn_Alex.pt')

In [ ]:
w = torch.load('Hahn_Alex.pt')

In [ ]:
[w0,w1,w2,w3,w4,w5,w6,w7] = w

In [ ]:
[w0,w1,w2,w3,w4,w5,w6,w7] = torch.load('Hahn_Alex.pt')

In [ ]:
img_t.shape,w0.shape

In [ ]:
img_t.shape

In [ ]:
img_t[0,:,:,:].shape

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
numpy_array = img_t[0,:,:,:].cpu().numpy()
numpy_array_transposed = numpy_array.transpose(1, 2, 0)
plt.imshow(numpy_array_transposed)
plt.show()

In [ ]:
def scale(img):
    # Normalize the NumPy array to the range [0, 1]
    max_value = img.max()
    min_value = img.min()
    normalized_array = (img - min_value) / (max_value - min_value)
    return normalized_array

In [ ]:
def tensor_plot(img_t,index=0):
    numpy_array = img_t[index,:,:,:].cpu().numpy()
    numpy_array_transposed = numpy_array.transpose(1, 2, 0)
    numpy_array_transposed = scale(numpy_array_transposed)
    plt.imshow(numpy_array_transposed)
    plt.show()

In [ ]:
tensor_plot(img_t)

In [ ]:
tensor_plot(w0,0)

In [ ]:
tensor_plot(w0,1)

In [ ]:
for i in range(64):
    tensor_plot(w0,i)

In [ ]:
w0.shape

In [ ]:
img_t[0,:,:,:]

In [ ]:
f0 = F.conv2d(img_t, w0, stride=4, padding=2)

In [ ]:
f0.shape

In [ ]:
tensor_plot(w0,0)

In [ ]:
i = 0
plt.imshow(f0[0,i,:,:].cpu().numpy())

In [ ]:
for i in range(64):
    tensor_plot(w0,i)
    plt.imshow(f0[0,i,:,:].cpu().numpy())
    plt.show()

In [ ]:
f1 = F.relu(f0)

In [ ]:
f1.shape

In [ ]:
f2 = F.max_pool2d(f1,kernel_size=3, stride=2, padding=0, dilation=1)

In [ ]:
f2.shape

In [ ]:
i = 0
plt.imshow(f0[0,i,:,:].cpu().numpy())
plt.colorbar()

In [ ]:
i = 0
plt.imshow(f1[0,i,:,:].cpu().numpy())
plt.colorbar()

In [ ]:
i = 0
plt.imshow(f2[0,i,:,:].cpu().numpy())
plt.colorbar()

In [ ]:
f3 = F.conv2d(f2, w1, stride=1, padding=2)

f4 = F.relu(f3)

f5 = F.max_pool2d(f4,kernel_size=3, stride=2, padding=0, dilation=1)

f6 = F.conv2d(f5, w2, stride=1, padding=1)

f7 = F.relu(f6)

f8 = F.conv2d(f7, w3, stride=1, padding=1)

f9 = F.relu(f8)

f10 = F.conv2d(f9, w4, stride=1, padding=1)

f11 = F.relu(f10)

f12 = F.max_pool2d(f11, kernel_size=3, stride=2, padding=0, dilation=1)

f13 = F.adaptive_avg_pool2d(f12,output_size=6).flatten()

f14 = F.linear(f13,w5)

f15 = F.relu(f14)

f16 = F.linear(f15,w6)

f17 = F.relu(f16)

f18 = F.linear(f17,w7)

out = f18.argmax().item()

In [ ]:
f18.shape

In [ ]:
plt.plot(f18.cpu().numpy())

In [ ]:
out

In [ ]:
labels

In [ ]:


labels[out]

In [ ]:
f18.argmax()

In [ ]:
labels[207]

In [ ]:
ls -l

In [ ]:
######
###
#

In [ ]:
def C1(x,w):
    return F.conv2d(x, w, stride=4, padding=2)

In [ ]:
def R(x):
    return F.relu(x)

In [ ]:
def M(x):
    return F.max_pool2d(x, kernel_size=3, stride=2, padding=0, dilation=1)

In [ ]:
def C2(x,w):
    return F.conv2d(x, w, stride=1, padding=2)

In [ ]:
def C3(x,w):
    return F.conv2d(x, w, stride=1, padding=1)

In [ ]:
def A(x):
    return F.adaptive_avg_pool2d(x,output_size=6).flatten()

In [ ]:
def L(x,w):
    return F.linear(x,w)

In [ ]:
f0 = C1(img_t, w0)

f1 = R(f0)

f2 = M(f1)

f3 = C2(f2, w1)

f4 = R(f3)

f5 = M(f4)

f6 = C3(f5,w2)

f7 = R(f6)

f8 = C3(f7,w3)

f9 = R(f8)

f10 = C3(f9,w4)

f11 = R(f10)

f12 = M(f11)

f13 = A(f12)

f14 = L(f13,w5)

f15 = R(f14)

f16 = L(f15,w6)

f17 = R(f16)

f18 = L(f17,w7)

out = f18.argmax().item()

labels[out]

In [ ]:
####################

In [ ]:
f0 = C1(img_t, w0)
f1 = R(f0)
f2 = M(f1)
f3 = C2(f2, w1)
f4 = R(f3)
f5 = M(f4)
f6 = C3(f5,w2)
f7 = R(f6)
f8 = C3(f7,w3)
f9 = R(f8)
f10 = C3(f9,w4)
f11 = R(f10)
f12 = M(f11)
f13 = A(f12)
f14 = L(f13,w5)
f15 = R(f14)
f16 = L(f15,w6)
f17 = R(f16)
f18 = L(f17,w7)

out = f18.argmax().item()

labels[out]

In [ ]:
f0 = C1(img_t, w0)
f1 = R(C1(img_t, w0))
f2 = M(R(C1(img_t, w0)))
f3 = C2(M(R(C1(img_t, w0))), w1)
f4 = R(C2(M(R(C1(img_t, w0))), w1))
f5 = M(R(C2(M(R(C1(img_t, w0))), w1)))
f6 = C3(M(R(C2(M(R(C1(img_t, w0))), w1))),w2)
f7 = R(C3(M(R(C2(M(R(C1(img_t, w0))), w1))),w2))
f8 = C3(R(C3(M(R(C2(M(R(C1(img_t, w0))), w1))),w2)),w3)
f9 = R(C3(R(C3(M(R(C2(M(R(C1(img_t, w0))), w1))),w2)),w3))
f10 = C3(R(C3(R(C3(M(R(C2(M(R(C1(img_t, w0))), w1))),w2)),w3)),w4)
f11 = R(C3(R(C3(R(C3(M(R(C2(M(R(C1(img_t, w0))), w1))),w2)),w3)),w4))
f12 = M(R(C3(R(C3(R(C3(M(R(C2(M(R(C1(img_t, w0))), w1))),w2)),w3)),w4)))
f13 = A(M(R(C3(R(C3(R(C3(M(R(C2(M(R(C1(img_t, w0))), w1))),w2)),w3)),w4))))
f14 = L(A(M(R(C3(R(C3(R(C3(M(R(C2(M(R(C1(img_t, w0))), w1))),w2)),w3)),w4)))),w5)
f15 = R(L(A(M(R(C3(R(C3(R(C3(M(R(C2(M(R(C1(img_t, w0))), w1))),w2)),w3)),w4)))),w5))
f16 = L(R(L(A(M(R(C3(R(C3(R(C3(M(R(C2(M(R(C1(img_t, w0))), w1))),w2)),w3)),w4)))),w5)),w6)
f17 = R(L(R(L(A(M(R(C3(R(C3(R(C3(M(R(C2(M(R(C1(img_t, w0))), w1))),w2)),w3)),w4)))),w5)),w6))
f18 = L(R(L(R(L(A(M(R(C3(R(C3(R(C3(M(R(C2(M(R(C1(img_t, w0))), w1))),w2)),w3)),w4)))),w5)),w6)),w7)

out = f18.argmax().item()

labels[out]

In [ ]:
f18 = L(R(L(R(L(A(M(R(C3(R(C3(R(C3(M(R(C2(M(R(C1(img_t,w0))),w1))),w2)),w3)),w4)))),w5)),w6)),w7)

out = f18.argmax().item()

labels[out]

In [ ]:
w0.shape,w1.shape

# Notes

In [ ]:
w1 = alexnet.features[0].weight.data.cpu().numpy()

In [ ]:
w1.shape

In [ ]:
alexnet

In [ ]:
for i in [0,3,6,8,10]:

    w = alexnet.features[i].weight.data.cpu().numpy()
    print(w.shape)
    plot(w[0,0,:,:])

In [ ]:
for i in [1,4,6]:

    w = alexnet.classifier[i].weight.data.cpu().numpy()
    print(w.shape)
    plot(w[:100,:100])

In [ ]:
import torch.nn.functional as F

In [ ]:
img_t

In [ ]:
F.conv2d()

In [ ]:
F.relu()

In [ ]:
F.max_pool2d(input, kernel_size, stride=None, padding=0, dilation=1)

In [ ]:
img_t.shape

In [ ]:
alexnet.features[0](img_t).cpu().detach().numpy().shape

In [ ]:
w0 = alexnet.features[0].weight.data
w1 = alexnet.features[3].weight.data
w2 = alexnet.features[6].weight.data
w3 = alexnet.features[8].weight.data
w4 = alexnet.features[10].weight.data
w5 = alexnet.classifier[1].weight.data
w6 = alexnet.classifier[4].weight.data
w7 = alexnet.classifier[6].weight.data

In [ ]:
w0.shape

In [ ]:
f0 = F.conv2d(img_t, w0, stride=4, padding=2)

In [ ]:
alexnet.classifier[0:7](alexnet.avgpool(alexnet.features[0:13](img_t)).flatten())

In [ ]:
alexnet.classifier[0:7](alexnet.avgpool(alexnet.features[0:13](img_t)).flatten()).argmax()

In [ ]:
labels[207]

In [ ]:
labels[alexnet.classifier[0:7](alexnet.avgpool(alexnet.features[0:13](img_t)).flatten()).argmax().item()]

In [ ]:
labels[alexnet.classifier[0:7](alexnet.avgpool(alexnet.features[1:13](f0)).flatten()).argmax().item()]

In [ ]:
labels[alexnet.classifier[0:7](F.adaptive_avg_pool2d(alexnet.features[1:13](f0),output_size=6).flatten()).argmax().item()]

In [ ]:
labels[alexnet.classifier[0:7](F.adaptive_avg_pool2d(alexnet.features[1:13](f0),output_size=6).flatten()).argmax().item()]

In [ ]:
f0 = F.conv2d(img_t, w0, stride=4, padding=2)

f1 = F.relu(f0)

f_conv = alexnet.features[2:13](f1)

fL = F.adaptive_avg_pool2d(f_conv,output_size=6).flatten()

labels[alexnet.classifier[0:7](fL).argmax().item()]

In [ ]:
########################

In [ ]:
f0 = F.conv2d(img_t, w0, stride=4, padding=2)

f1 = F.relu(f0)

f2 = F.max_pool2d(f1,kernel_size=3, stride=2, padding=0, dilation=1)

f_conv = alexnet.features[3:13](f2)

fL = F.adaptive_avg_pool2d(f_conv,output_size=6).flatten()

out = alexnet.classifier[0:7](fL)

labels[out.argmax().item()]

In [ ]:
alexnet

In [ ]:
f0 = F.conv2d(img_t, w0, stride=4, padding=2)

f1 = F.relu(f0)

f2 = F.max_pool2d(f1,kernel_size=3, stride=2, padding=0, dilation=1)

f3 = F.conv2d(f2, w1, stride=1, padding=2)

f4 = F.relu(f3)



f_conv = alexnet.features[5:13](f4)

fL = F.adaptive_avg_pool2d(f_conv,output_size=6).flatten()

out = alexnet.classifier[0:7](fL)

labels[out.argmax().item()]

In [ ]:
####
####

In [ ]:
f0 = F.conv2d(img_t, w0, stride=4, padding=2)

f1 = F.relu(f0)

f2 = F.max_pool2d(f1,kernel_size=3, stride=2, padding=0, dilation=1)

f3 = F.conv2d(f2, w1, stride=1, padding=2)

f4 = F.relu(f3)

f5 = F.max_pool2d(f4,kernel_size=3, stride=2, padding=0, dilation=1)



f_conv = alexnet.features[6:13](f5)

fL = F.adaptive_avg_pool2d(f_conv,output_size=6).flatten()

out = alexnet.classifier[0:7](fL)

labels[out.argmax().item()]

In [ ]:
##################
##################

In [ ]:
f0 = F.conv2d(img_t, w0, stride=4, padding=2)

f1 = F.relu(f0)

f2 = F.max_pool2d(f1,kernel_size=3, stride=2, padding=0, dilation=1)

f3 = F.conv2d(f2, w1, stride=1, padding=2)

f4 = F.relu(f3)

f5 = F.max_pool2d(f4,kernel_size=3, stride=2, padding=0, dilation=1)

f6 = F.conv2d(f5, w2, stride=1, padding=1)


f_conv = alexnet.features[7:13](f6)

fL = F.adaptive_avg_pool2d(f_conv,output_size=6).flatten()

out = alexnet.classifier[0:7](fL)

labels[out.argmax().item()]

In [ ]:
############

In [ ]:
f0 = F.conv2d(img_t, w0, stride=4, padding=2)

f1 = F.relu(f0)

f2 = F.max_pool2d(f1,kernel_size=3, stride=2, padding=0, dilation=1)

f3 = F.conv2d(f2, w1, stride=1, padding=2)

f4 = F.relu(f3)

f5 = F.max_pool2d(f4,kernel_size=3, stride=2, padding=0, dilation=1)

f6 = F.conv2d(f5, w2, stride=1, padding=1)

f7 = F.relu(f6)

f8 = F.conv2d(f7, w3, stride=1, padding=1)

f9 = F.relu(f8)

f10 = F.conv2d(f9, w4, stride=1, padding=1)



f_conv = alexnet.features[11:13](f10)

fL = F.adaptive_avg_pool2d(f_conv,output_size=6).flatten()

out = alexnet.classifier[0:7](fL)

labels[out.argmax().item()]

In [ ]:
alexnet

In [ ]:
######### ####

In [ ]:
f0 = F.conv2d(img_t, w0, stride=4, padding=2)

f1 = F.relu(f0)

f2 = F.max_pool2d(f1,kernel_size=3, stride=2, padding=0, dilation=1)

f3 = F.conv2d(f2, w1, stride=1, padding=2)

f4 = F.relu(f3)

f5 = F.max_pool2d(f4,kernel_size=3, stride=2, padding=0, dilation=1)

f6 = F.conv2d(f5, w2, stride=1, padding=1)

f7 = F.relu(f6)

f8 = F.conv2d(f7, w3, stride=1, padding=1)

f9 = F.relu(f8)

f10 = F.conv2d(f9, w4, stride=1, padding=1)

f11 = F.relu(f10)


f_conv = alexnet.features[12:13](f11)

fL = F.adaptive_avg_pool2d(f_conv,output_size=6).flatten()

out = alexnet.classifier[0:7](fL)

labels[out.argmax().item()]

In [ ]:
f0 = F.conv2d(img_t, w0, stride=4, padding=2)

f1 = F.relu(f0)

f2 = F.max_pool2d(f1,kernel_size=3, stride=2, padding=0, dilation=1)

f3 = F.conv2d(f2, w1, stride=1, padding=2)

f4 = F.relu(f3)

f5 = F.max_pool2d(f4,kernel_size=3, stride=2, padding=0, dilation=1)

f6 = F.conv2d(f5, w2, stride=1, padding=1)

f7 = F.relu(f6)

f8 = F.conv2d(f7, w3, stride=1, padding=1)

f9 = F.relu(f8)

f10 = F.conv2d(f9, w4, stride=1, padding=1)

f11 = F.relu(f10)

f12 = F.max_pool2d(f11, kernel_size=3, stride=2, padding=0, dilation=1)


f_conv = alexnet.features[13:13](f12)

fL = F.adaptive_avg_pool2d(f_conv,output_size=6).flatten()

out = alexnet.classifier[0:7](fL)

labels[out.argmax().item()]

In [ ]:
f0 = F.conv2d(img_t, w0, stride=4, padding=2)

f1 = F.relu(f0)

f2 = F.max_pool2d(f1,kernel_size=3, stride=2, padding=0, dilation=1)

f3 = F.conv2d(f2, w1, stride=1, padding=2)

f4 = F.relu(f3)

f5 = F.max_pool2d(f4,kernel_size=3, stride=2, padding=0, dilation=1)

f6 = F.conv2d(f5, w2, stride=1, padding=1)

f7 = F.relu(f6)

f8 = F.conv2d(f7, w3, stride=1, padding=1)

f9 = F.relu(f8)

f10 = F.conv2d(f9, w4, stride=1, padding=1)

f11 = F.relu(f10)

f12 = F.max_pool2d(f11, kernel_size=3, stride=2, padding=0, dilation=1)

f13 = F.adaptive_avg_pool2d(f12,output_size=6).flatten()

out = alexnet.classifier[0:7](f13)

labels[out.argmax().item()]

In [ ]:
## ## ## ##

In [ ]:
f0 = F.conv2d(img_t, w0, stride=4, padding=2)

f1 = F.relu(f0)

f2 = F.max_pool2d(f1,kernel_size=3, stride=2, padding=0, dilation=1)

f3 = F.conv2d(f2, w1, stride=1, padding=2)

f4 = F.relu(f3)

f5 = F.max_pool2d(f4,kernel_size=3, stride=2, padding=0, dilation=1)

f6 = F.conv2d(f5, w2, stride=1, padding=1)

f7 = F.relu(f6)

f8 = F.conv2d(f7, w3, stride=1, padding=1)

f9 = F.relu(f8)

f10 = F.conv2d(f9, w4, stride=1, padding=1)

f11 = F.relu(f10)

f12 = F.max_pool2d(f11, kernel_size=3, stride=2, padding=0, dilation=1)

f13 = F.adaptive_avg_pool2d(f12,output_size=6).flatten()

f14 = F.linear(f13,w5)

f15 = F.relu(f14)

f16 = F.linear(f15,w6)

f17 = F.relu(f16)

out = F.linear(f17,w7)


labels[out.argmax().item()]

In [ ]:
! wget -q "https://tinyurl.com/HahnImports"
exec(open("HahnImports").read())

##[Tensorspace](https://tensorspace.org/html/playground/alexnet.html)

![alt text](https://miro.medium.com/max/750/1*IlzW43-NtJrwqtt5Xy3ISA.jpeg)

![alt text](https://d3i71xaburhd42.cloudfront.net/38211dc39e41273c0007889202c69f841e02248a/2-Figure1-1.png)

![alt text](https://i.stack.imgur.com/XBWc3.png)

![alt text](https://cv-tricks.com/wp-content/uploads/2017/03/alexnet_small-1.png)

![alt text](https://www.learnopencv.com/wp-content/uploads/2017/11/convolution-example-matrix.gif)

<img src="https://production-media.paperswithcode.com/methods/MaxpoolSample2.png">

<img src=https://ujwlkarn.files.wordpress.com/2016/08/giphy.gif width="900">


In [ ]:
def video_stream():
  js = Javascript('''
    var video;
    var div = null;
    var stream;
    var captureCanvas;
    var imgElement;
    var labelElement;

    var pendingResolve = null;
    var shutdown = false;

    function removeDom() {
       stream.getVideoTracks()[0].stop();
       video.remove();
       div.remove();
       video = null;
       div = null;
       stream = null;
       imgElement = null;
       captureCanvas = null;
       labelElement = null;
    }

    function onAnimationFrame() {
      if (!shutdown) {
        window.requestAnimationFrame(onAnimationFrame);
      }
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL('image/jpeg', 0.8)
        }
        var lp = pendingResolve;
        pendingResolve = null;
        lp(result);
      }
    }

    async function createDom() {
      if (div !== null) {
        return stream;
      }

      div = document.createElement('div');
      div.style.border = '2px solid black';
      div.style.padding = '3px';
      div.style.width = '100%';
      div.style.maxWidth = '600px';
      document.body.appendChild(div);


      video = document.createElement('video');
      video.style.display = 'block';
      video.width = div.clientWidth - 6;
      video.setAttribute('playsinline', '');
      video.onclick = () => { shutdown = true; };
      stream = await navigator.mediaDevices.getUserMedia(
          {video: { facingMode: "environment"}});
      div.appendChild(video);

      imgElement = document.createElement('img');
      imgElement.style.position = 'absolute';
      imgElement.style.zIndex = 1;
      imgElement.onclick = () => { shutdown = true; };
      div.appendChild(imgElement);

      const instruction = document.createElement('div');
      instruction.innerHTML =
          '<span style="blue: red; font-weight: bold;">' +
          'click here to stop the video</span>';
      div.appendChild(instruction);
      instruction.onclick = () => { shutdown = true; };

      video.srcObject = stream;
      await video.play();

      captureCanvas = document.createElement('canvas');
      captureCanvas.width = 640;
      captureCanvas.height = 480;
      window.requestAnimationFrame(onAnimationFrame);

      return stream;
    }
    async function stream_frame() {
      if (shutdown) {
        removeDom();
        shutdown = false;
        return '';
      }

      var preCreate = Date.now();
      stream = await createDom();

      var preShow = Date.now();




      var preCapture = Date.now();
      var result = await new Promise(function(resolve, reject) {
        pendingResolve = resolve;
      });
      shutdown = false;

      return {'create': preShow - preCreate,
              'show': preCapture - preShow,
              'capture': Date.now() - preCapture,
              'img': result};
    }
    ''')

  display(js)
def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

In [ ]:
def scale(x):
    return (x - np.min(x))/np.max(x)
def scale1(x):
    return (x-np.min(x))/(np.max(x)-np.min(x))

In [ ]:
def plot(x):
    fig, ax = plt.subplots()
    im = ax.imshow(x,cmap='gray')
    ax.axis('off')
    fig.set_size_inches(10, 10)
    plt.show()

In [ ]:
net = alexnet(pretrained=True).cuda(0)

In [ ]:
normalize = transforms.Normalize(
   mean=[0.485, 0.456, 0.406],
   std=[0.229, 0.224, 0.225]
)
preprocess = transforms.Compose([
   transforms.Resize(256),
   transforms.CenterCrop(224),
   transforms.ToTensor(),
   normalize
])

In [ ]:
filename = take_photo()
print('Saved to {}'.format(filename))

In [ ]:
im = imageio.imread('https://www.medicalnewstoday.com/content/images/articles/322/322868/golden-retriever-puppy.jpg')

In [ ]:
im = imageio.imread('photo.jpg')

In [ ]:
plot(im)

In [ ]:
image = Image.fromarray(im) #convert to pil

In [ ]:
img_tensor = preprocess(image)

In [ ]:
img_tensor.shape

In [ ]:
img_tensor = img_tensor[None,:,:,:]

In [ ]:
img_tensor.shape

In [ ]:
img_variable = torch.tensor(img_tensor).cuda(0)

In [ ]:
img_variable.shape

In [ ]:
out = net(img_variable)

In [ ]:
%%timeit
out = net(img_variable)

In [ ]:
out.shape

In [ ]:
out.cpu().data.numpy().argmax()

In [ ]:
label_index = out.cpu().data.numpy().argmax()

In [ ]:
label_index

In [ ]:
top_list = np.flip(np.argsort(out.cpu().data.numpy())[0][-10:])

In [ ]:
top_list

In [ ]:
LABELS_URL = 'https://s3.amazonaws.com/mlpipes/pytorch-quick-start/labels.json'

In [ ]:
labels = {int(key):value for (key, value) in requests.get(LABELS_URL).json().items()}

In [ ]:
labels

In [ ]:
labels[800]

In [ ]:
print(labels[label_index])

In [ ]:
for i in range(10):
    print(labels[top_list[i]])

In [ ]:
net

In [ ]:
summary(net, (3, 224, 224))

In [ ]:
out = net.features[0](img_variable).cpu().detach().numpy()

In [ ]:
plot(out[0,12,:,:])

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:6](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
im = imageio.imread('http://bocasurfcam.com/most_recent_image.php')

In [ ]:
plot(im)

In [ ]:
def load_im(im):
    image = Image.fromarray(im) #convert to pil
    img_tensor = preprocess(image)
    img_tensor = img_tensor.unsqueeze_(0)
    img_variable = torch.tensor(img_tensor).cuda(0)
    return img_variable

In [ ]:
out = net(load_im(im))

In [ ]:
def inference(im):
    out = net(load_im(im))
    top_list = np.flip(np.argsort(out.cpu().data.numpy())[0][-10:])
    for i in range(10):
        print(labels[top_list[i]])

In [ ]:
inference(im)

In [ ]:
w1 = net.features[0].weight.data.cpu().numpy()

In [ ]:
w1.shape

In [ ]:
w1 = np.transpose(w1,(0,2,3,1))

In [ ]:
w1.shape

In [ ]:
Wp = (w1 - np.mean(w1))/np.std(w1)

In [ ]:
M = montage(Wp, multichannel=True)

In [ ]:
M.shape

In [ ]:
plot(Wp[0,:,:,:])

In [ ]:
plot(np.random.random((11,11,3)))

In [ ]:
def plot(x):

    if torch.is_tensor(x):
        x = x.cpu().detach().numpy()


    if len(x.shape)>3:
        if x.shape[3] == 3:
            x = montage(x, multichannel=True)
        else:
            x = montage(x, multichannel=False)

    fig, ax = plt.subplots()
    im = ax.imshow(x,cmap='gray')
    ax.axis('off')
    fig.set_size_inches(10, 10)
    plt.show()

In [ ]:
plot(Wp)

In [ ]:
plot(w1)

In [ ]:
plot(scale1(montage(np.transpose(net.features[0].weight.data.cpu().numpy(),(0,2,3,1)), multichannel=True)))

In [ ]:
plot(scale(montage(np.transpose(net.features[0].weight.data.cpu().numpy(),(0,2,3,1)), multichannel=True)))

In [ ]:
img_variable.flatten().shape

In [ ]:
out = net.features[0](img_variable).cpu().detach().numpy()

In [ ]:
out.shape

In [ ]:
plot(out[0,0,:,:])

In [ ]:
img_variable.shape

In [ ]:
3*224*224

In [ ]:
64*55*55

In [ ]:
out = np.squeeze(out)

In [ ]:
plot(montage(out))

In [ ]:
plot(scale(np.transpose(net.features[0].weight.data.cpu().numpy(),(0,2,3,1))))

In [ ]:
def plot2(x):
    plot(montage(np.squeeze(x.cpu().detach().numpy())))

In [ ]:
fmaps = net.features[0:1](img_variable)

In [ ]:
plot2(fmaps)

In [ ]:
for i in range(12):
    fmaps = net.features[0:i](img_variable)
    plot2(fmaps)

In [ ]:
plot2(net.avgpool(net.features[0:13](img_variable)))

In [ ]:
net.features[0:13](img_variable).cpu().detach().numpy().shape

In [ ]:
net

In [ ]:
net.avgpool(net.features[0:13](img_variable)).shape

In [ ]:
net.avgpool(net.features[0:13](img_variable)).flatten().shape[0]

In [ ]:
plt.plot(np.arange(9216),net.avgpool(net.features[0:13](img_variable)).flatten().cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
9216 / (3*224*224) # percentage of numbers left, vs original pixels

In [ ]:
net.classifier[0](net.avgpool(net.features[0:13](img_variable)).flatten()).shape

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:2](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:3](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:5](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
im.shape

In [ ]:
1920*2560*3

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:6](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
4000/14000000

In [ ]:
plt.plot(np.arange(1000),net.classifier[0:7](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
out = net.classifier[0:7](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy()

In [ ]:
def softmax(x):

    s = np.exp(x)

    s = s / np.sum(s)

    return s

In [ ]:
out = softmax(out)

In [ ]:
plt.plot(np.arange(1000),out)
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
label_index = out.argmax()

In [ ]:
label_index

In [ ]:
top_list = np.flip(np.argsort(out))[0:10]

In [ ]:
top_list

In [ ]:
for i in range(10):
    print(labels[top_list[i]])

In [ ]:
def photo():
    return jsob_to_image(video_frame()["img"])

In [ ]:
video_stream()

while True:

    img = photo()

    print(img.shape)

    out = net(load_im(im))
    plt.plot(np.arange(1000),out[0].cpu().detach().numpy())
    plt.show()

    inference(img)

In [ ]:
img